# Fashion-MNIST Frequent Itemset Mining

Ứng dụng Eclat cho cấu hình đã chốt **B0 / K=32 / seed=123 / S0** trên 70.000 giao dịch. Notebook này chuẩn bị môi trường, kiểm tra nguồn đã khóa, chạy P1 smoke, khởi động supervisor ngoài kernel cho sáu ngưỡng production và dựng phân tích từ các kết quả đã commit.

**Nguồn mining chỉ đọc:** `results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123/transactions.jsonl`. P1 chỉ dùng toy data; production chỉ được khởi chạy ở cell P2.

In [1]:
# P0 — Môi trường và nguồn dữ liệu đã khóa
using Pkg, SHA, TOML, Dates
function find_repo_root(start)
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src/EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy gốc repository từ $(start)")
        current = parent
    end
end
REPO_ROOT = find_repo_root(pwd())
Pkg.activate(REPO_ROOT)
using EclatFIM

INPUT_PATH = joinpath(REPO_ROOT, "results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123/transactions.jsonl")
OUTPUT_DIR = joinpath(REPO_ROOT, "results/fashion/fim/B0_K32_seed123_S0")
PYTHON = Sys.which("python3")
isnothing(PYTHON) && error("Không tìm thấy python3")
JULIA = realpath(first(Base.julia_cmd().exec))
SUPERVISOR = joinpath(REPO_ROOT, "scripts/fashion_fim/supervisor.py")
SMOKE = joinpath(REPO_ROOT, "scripts/fashion_fim/smoke.py")
ANALYSIS = joinpath(REPO_ROOT, "scripts/fashion_fim/analysis.py")
LOCKED_INPUT_SHA256 = "b078e485064bfdacea4b5faa5f027efe7123af32a2db2c27acf415d6b1dd7d70"
LOCKED_ECLAT_SHA256 = "0a480452321e9024aaeced4b6334500ad3a9795c947aa03edc3f7d7ab227c4ed"

@assert isfile(INPUT_PATH) "Không tìm thấy transactions S0 đã khóa"
input_sha256 = bytes2hex(sha256(read(INPUT_PATH)))
engine_sha256 = bytes2hex(sha256(read(joinpath(REPO_ROOT, "src/algorithm/EclatBaseline.jl"))))
@assert input_sha256 == LOCKED_INPUT_SHA256 "SHA-256 input không khớp; dừng trước mining"
@assert engine_sha256 == LOCKED_ECLAT_SHA256 "Eclat core đổi sau khi rà soát; dừng để kiểm tra"

mkpath(joinpath(OUTPUT_DIR, "runtime"))
pkg_versions = sort(["$(info.name)=$(info.version)" for info in values(Pkg.dependencies()) if info.version !== nothing])
runtime_env = Dict(
    "julia_executable" => JULIA,
    "julia_version" => string(VERSION),
    "julia_threads" => Threads.nthreads(),
    "julia_depot_path" => join(DEPOT_PATH, ":"),
    "julia_load_path" => join(LOAD_PATH, ":"),
    "home_path" => homedir(),
    "process_path" => get(ENV, "PATH", ""),
    "julia_package_versions" => pkg_versions,
    "cpu_model" => Sys.CPU_NAME,
    "memory_available_bytes" => Sys.free_memory(),
)
open(joinpath(OUTPUT_DIR, "runtime/runtime_env.toml"), "w") do io
    TOML.print(io, runtime_env)
end
(; julia=string(VERSION), julia_executable=JULIA, threads=Threads.nthreads(),
   input_sha256, engine_sha256, input_path=INPUT_PATH, output_dir=OUTPUT_DIR, eclat_api=isdefined(EclatFIM, :mine))


  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


(julia = "1.13.3", julia_executable = "/home/fernir/.julia/juliaup/julia-1.13.3+0.x64.linux.gnu/bin/julia", threads = 1, input_sha256 = "b078e485064bfdacea4b5faa5f027efe7123af32a2db2c27acf415d6b1dd7d70", engine_sha256 = "0a480452321e9024aaeced4b6334500ad3a9795c947aa03edc3f7d7ab227c4ed", input_path = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123/transactions.jsonl", output_dir = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0", eclat_api = true)

## P1 — Smoke và khởi tạo
Cell tiếp theo chỉ chạy smoke trên toy database, sau đó kiểm tra toàn bộ 70.000 dòng input và tạo sáu sidecar trạng thái ban đầu là `NOT_RUN`. Nó không chạy mining production. Các giới hạn P2 được ghi rõ: 3.600 giây/run, 3.072 MiB cho worker và 8 GiB cho một JSONL.

In [7]:
# P1 — Smoke toy + preflight/init (không chạy sáu ngưỡng production)
run(Cmd([PYTHON, SMOKE, REPO_ROOT, JULIA]))
run(Cmd([
    PYTHON, SUPERVISOR, "--init",
    "--repo-root", REPO_ROOT,
    "--input-path", INPUT_PATH,
    "--output-dir", OUTPUT_DIR,
    "--runtime-env", joinpath(OUTPUT_DIR, "runtime/runtime_env.toml"),
    "--wall-time-seconds-per-run", "3600",
    "--memory-limit-mb-per-worker", "3072",
    "--output-limit-bytes", "8589934592",
    "--budget-revision", "b1",
]))


{
  "analysis": "PASS",
  "completed_runs": 3,
  "unique_patterns": 3,
  "representatives": 3,
  "threshold": 0.9,
  "relaxed": true,
  "files": [
    "analysis/count_by_length.csv",
    "analysis/pattern_support_summary.csv",
    "analysis/master_patterns.jsonl",
    "analysis/run_statistics.csv",
    "analysis/pattern_statistics.md",
    "analysis/representative_patterns.csv",
    "analysis/minsup_vs_itemset_count.svg",
    "analysis/minsup_vs_runtime.svg",
    "analysis/itemset_length_distribution.svg",
    "figures/minsup_vs_itemset_count.png",
    "figures/minsup_vs_runtime.png",
    "figures/itemset_lengths_by_minsup.png"
  ]
}
PASS Julia loader keeps row/TID/image mapping
PASS public Eclat API returns expected Dict and optional-limit semantics
PASS toy full JSONL output is sorted and support fractions are transaction-based
PASS COMPLETED_ZERO writes a 0-byte exact staging file
PASS output_limit_bytes stops before overflow
PASS systemd user service enforces MemoryMax and MemorySw

Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/supervisor.py --init --repo-root /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining --input-path /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123/transactions.jsonl --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0 --runtime-env /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0/runtime/runtime_env.toml --wall-time-seconds-per-run 3600 --memory-limit-mb-per-worker 3072 --output-limit-bytes 8589934592 --budget-revision b1`, ProcessExited(0))

## P2 — Production qua MCP
Chỉ chạy cell launch **một lần** sau khi kernel Julia đã restart và cell P0/P1 hoàn tất. Supervisor chạy ngoài notebook kernel; sáu minsup tuần tự là 80%, 60%, 40%, 20%, 10%, 5%. Có thể đóng hoặc restart notebook sau khi service đã xác nhận khởi chạy. Dùng cell status để theo dõi.

In [3]:
# P2 — Khởi chạy supervisor ngoài kernel
run(Cmd([PYTHON, SUPERVISOR, "--launch", "--output-dir", OUTPUT_DIR]))


{"launched": true, "unit": "fim-supervisor-b1-b078e485.service", "log_path": "results/fashion/fim/B0_K32_seed123_S0/report/supervisor.log", "acknowledgement": ""}


Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/supervisor.py --launch --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0`, ProcessExited(0))

## P2 status và P3 analysis
Chạy status để xem tiến độ. Chạy analysis cell khi cả sáu worker đã kết thúc; các run có giới hạn tài nguyên được báo như partial và không được coi exact. P3 chỉ đọc output đã commit, không gọi Eclat.

In [5]:
# P2 — Status snapshot
run(Cmd([PYTHON, SUPERVISOR, "--status", "--output-dir", OUTPUT_DIR]))


{
  "supervisor_unit": "fim-supervisor-b1-b078e485.service",
  "supervisor": {
    "ActiveState": "inactive",
    "SubState": "dead",
    "MainPID": "0",
    "Result": "success",
    "ExecMainStartTimestampMonotonic": "0",
    "ExecMainExitTimestampMonotonic": "0",
    "ExecMainCode": "0",
    "ExecMainStatus": "0",
    "ControlGroup": "",
    "MemoryCurrent": "[not set]",
    "MemoryPeak": "[not set]",
    "MemoryMax": "infinity",
    "MemorySwapMax": "infinity"
  },
  "runs": [
    {
      "run_id": "minsup_80_b078e485_0a480452_b1_a01",
      "configuration": "B0_K32_seed123_S0",
      "minsup_fraction": "0.8",
      "minsup_count": "56000",
      "status": "COMPLETED_ZERO",
      "reason_code": "",
      "total_frequent_itemsets": "0",
      "max_itemset_length": "0",
      "observed_partial_itemsets": "0",
      "runtime_seconds": "6.137632270998438",
      "peak_rss_mb": "444.8359375",
      "output_size_bytes": "0",
      "engine_revision": "994edaa25972e85acb24d604441ef95ac62ab3

Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/supervisor.py --status --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0`, ProcessExited(0))

## P3 — Exact analysis
Chỉ chạy khi các sidecar đều ở trạng thái kết thúc. Script kiểm tra checksum/read-back của mọi output exact, sau đó dựng thống kê, catalog, representatives và biểu đồ từ các kết quả đã commit.

In [8]:
# P3 — Hậu kiểm output đã commit và phân tích (không chạy lại Eclat)
manifest = TOML.parsefile(joinpath(OUTPUT_DIR, "report/experiment_manifest.toml"))
canonical_runs = values(manifest["canonical_runs"])
run_statuses = [
    TOML.parsefile(joinpath(OUTPUT_DIR, "mining/runs", run_id, "status.toml"))["status"]
    for run_id in canonical_runs
]
active_statuses = filter(s -> s in ("LAUNCHING", "STARTING", "RUNNING", "QUEUED"), run_statuses)
@assert isempty(active_statuses) "Supervisor vẫn đang chạy; kiểm tra status và thử lại P3 sau"
AUDIT = joinpath(REPO_ROOT, "scripts/fashion_fim/audit_outputs.py")
run(Cmd([PYTHON, AUDIT, "--output-dir", OUTPUT_DIR]))
run(Cmd([PYTHON, ANALYSIS, "--output-dir", OUTPUT_DIR, "--target", "16"]))


{
  "status": "PASS",
  "sampled_itemsets": 77,
  "sample_mismatch_count": 0,
  "cross_threshold_pairs": 15,
  "cross_threshold_mismatch_count": 0,
  "sidecars_reconciled": true,
  "artifact": "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0/analysis/independent_output_audit.json"
}
{
  "analysis": "PASS",
  "completed_runs": 6,
  "unique_patterns": 7228,
  "representatives": 16,
  "threshold": 0.8,
  "relaxed": false,
  "files": [
    "analysis/count_by_length.csv",
    "analysis/pattern_support_summary.csv",
    "analysis/master_patterns.jsonl",
    "analysis/run_statistics.csv",
    "analysis/pattern_statistics.md",
    "analysis/representative_patterns.csv",
    "analysis/minsup_vs_itemset_count.svg",
    "analysis/minsup_vs_runtime.svg",
    "analysis/itemset_length_distribution.svg",
    "figures/minsup_vs_itemset_count.png",
    "figures/minsup_vs_runtime.png",
    "figures/itemset_lengths_by_min

Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/analysis.py --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0 --target 16`, ProcessExited(0))

## P5 — Báo cáo sau interpretation
Sau khi chạy xong notebook **FIM Visual Interpretation** trong kernel Julia sạch, quay lại và tạo report/handoff. MCP execution log chỉ được ghi bằng chứng quan sát thật; không tự thêm marker PASS.

In [12]:
# P5 — Report/audit (chạy sau notebook interpretation)
const REPORT = joinpath(REPO_ROOT, "scripts/fashion_fim/report.py")
run(Cmd([PYTHON, REPORT, "--output-dir", OUTPUT_DIR]))


{
  "axes": {
    "mining": "PASS",
    "execution_mcp": "PASS",
    "interpretation": "PASS",
    "report_audit": "PASS"
  },
  "overall": "COMPLETE",
  "reason_codes": [],
  "report_audit": "PASS",
  "report": "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0/report/fashion_fim_application_report.md"
}


Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/report.py --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0`, ProcessExited(0))